# 23 — Greedy MoE phase 3c: balanced SupCon
Run the class-balanced-denominator SupCon loss on seeds 0–2 at the selected weight while retaining legacy sampling and CE weighting.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER='selimsidan'; GITHUB_REPO='dataset_moe_nids'; GITHUB_BRANCH='main'; GITHUB_SECRET_NAME='GITHUB_TOKEN'
DRIVE_DATA_DIR='/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
PREFIX='nfv3_4way_greedy_v1'; PHASE='3c'; EXECUTE=False; MAX_NEW_RUNS=None; RUN_TESTS=True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive'); token=userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode(); git_env=os.environ|{'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.https://github.com/.extraheader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}'}
repo=Path('/content')/GITHUB_REPO; url=f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH] if (repo/'.git').is_dir() else ['git','clone','--branch',GITHUB_BRANCH,'--single-branch',url,str(repo)],env=git_env,check=True)
git_env.clear(); token=auth=None; os.chdir(repo)
os.environ.update(NIDS_DRIVE_BASE=DRIVE_DATA_DIR,NIDS_OUTPUT_DIR=DRIVE_OUTPUT_DIR,NIDS_SCRATCH_DIR='/content/dataset_moe_nids_scratch')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())

## Preflight, dry run, and optional execution

In [ ]:
import torch
from training.ablation_matrix import expected_evaluated_run_count, load_study
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
study=load_study('config/greedy_moe_study.yaml'); print('GPU:',torch.cuda.get_device_name(0),'| planned runs:',expected_evaluated_run_count(study))
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q','tests/test_ablation_study.py','tests/test_representation_losses.py'],check=True)
command=[sys.executable,'-u','-m','training.ablation_matrix','--phase',PHASE,'--prefix',PREFIX]; subprocess.run(command,check=True,env=os.environ.copy())
if EXECUTE:
    execute_command=[*command,'--execute']
    if MAX_NEW_RUNS is not None: execute_command += ['--max-new-runs',str(MAX_NEW_RUNS)]
    subprocess.run(execute_command,check=True,env=os.environ.copy())

## Review representation coverage, state, and summaries

In [ ]:
import pandas as pd
from IPython.display import display
summary_dir=Path(DRIVE_OUTPUT_DIR)/'results'/f'{PREFIX}_summary'; state_path=summary_dir/'study_state.json'
if state_path.is_file(): print(json.dumps(json.loads(state_path.read_text()).get('decisions',{}),indent=2))
for path in sorted(summary_dir.glob(f'Ablation_Phase{PHASE}_*.csv')): print(path.name); display(pd.read_csv(path))
coverage=list((Path(DRIVE_OUTPUT_DIR)/'results').glob(f'{PREFIX}_balanced_supcon_*/Representation_Batch_Coverage.csv'))
for path in coverage: print(path.parent.name); display(pd.read_csv(path).sort_values('valid_anchor_fraction'))
print('Safe to rerun until phase status is complete:',summary_dir)